# BERNN minimal trainers examples

This notebook contains 4 short examples using in-repo data (`data/benchmark/intensities.csv`):

1. `TrainAEClassifierHoldout` with `pools=False`
2. `TrainAEClassifierHoldout` with `pools=True`
3. `TrainAEThenClassifierHoldout` with `pools=False`
4. `TrainAEThenClassifierHoldout` with `pools=True`

Important parameters shown in this notebook: `optimize_hyperparams`, `n_trials`, `n_repeats`, `n_layers`, `layer1`, `warmup`, `n_epochs`, `dloss`, `device`, `scaler`, `bs`.

In [ ]:
from pathlib import Path

import pandas as pd

from bernn import TrainAEClassifierHoldout, TrainAEThenClassifierHoldout
from bernn.config.training_config import TrainingConfig

csv_path = Path('../data/benchmark/intensities.csv')
if not csv_path.exists():
    raise FileNotFoundError(f'Missing dataset: {csv_path.resolve()}')

df = pd.read_csv(csv_path)
# BERNN convention: col0=sample id, col1=label, col2=batch, remaining=features
X = df.iloc[:, 3:]
y = df.iloc[:, 1].to_numpy()
batches = df.iloc[:, 2].to_numpy()

# Tiny split for demonstration only
split = max(8, int(0.8 * len(df)))
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y[:split], y[split:]
batches_train, batches_test = batches[:split], batches[split:]

print('X_train:', X_train.shape, 'X_test:', X_test.shape)

In [ ]:
# Keep this tiny for a quick smoke run. Increase for real training.
bernn_config = TrainingConfig(
    optimize_hyperparams=False,
    n_trials=1,
    n_repeats=1,
    n_layers=1,
    layer1=128,
    warmup=1,
    n_epochs=2,
    dloss='inverseTriplet',
    device='cpu',
    scaler='standard',
    bs=16,
)

In [ ]:
examples = [
    ('classifier_no_pool', TrainAEClassifierHoldout, False),
    ('classifier_with_pool', TrainAEClassifierHoldout, True),
    ('ae_then_classifier_no_pool', TrainAEThenClassifierHoldout, False),
    ('ae_then_classifier_with_pool', TrainAEThenClassifierHoldout, True),
]

all_preds = {}

for name, trainer_cls, pools in examples:
    trainer = trainer_cls(config=bernn_config, pools=pools, log_metrics=True, keep_models=False)

    # Train and predict in one call
    _ = trainer.fit_predict(
        X_train,
        y_train,
        X_test=X_test,
        y_test=y_test,
        groups_train=batches_train,
        groups_test=batches_test,
        cross_validation=False,
        cross_test=False,
    )

    # Decode predictions back to original labels
    preds = trainer.predict(X_test)
    all_preds[name] = preds
    print(name, 'done, preds shape:', getattr(preds, 'shape', type(preds)))